# Notebook Colab — Entraînement detection_fissures
Ce notebook guide l'installation, la préparation des données, l'entraînement et l'export du modèle Mask R-CNN pour le projet `detection_fissures`.

**Avant de commencer** : remplace `GITHUB_REPO_URL` par l'URL de ton dépôt ou utilise la copie depuis Drive.

## 1) Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Cloner le dépôt (ou copier depuis Drive)

In [ ]:
# Option A — cloner depuis GitHub (remplace l'URL)
!git clone <GITHUB_REPO_URL> detection_fissures || true
%cd /content/detection_fissures || true

# Option B — copier depuis Drive si tu as déjà uploadé le projet
# %cp -r /content/drive/MyDrive/chemin/vers/detection_fissures /content/ || true
# %cd /content/detection_fissures || true

## 3) Créer dossier de sorties sur Drive et lier à `sorties/`

In [ ]:
# Crée un dossier sur Drive pour stocker checkpoints et journaux
!mkdir -p /content/drive/MyDrive/detection_fissures_sorties
# Supprime l'ancien lien local si nécessaire et crée un lien symbolique
!rm -rf sorties || true
!ln -s /content/drive/MyDrive/detection_fissures_sorties sorties || true
!ls -la

## 4) Installer les dépendances (UV recommandé)

In [ ]:
# Option UV (si tu veux utiliser uv/astral)
# curl -LsSf https://astral.sh/uv/install.sh | sh
# uv sync

# Option pip — installe PyTorch (adapte la roue CUDA si nécessaire)
!pip install -U pip setuptools wheel
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 || true
!pip install opencv-python pycocotools albumentations tqdm matplotlib seaborn
# Si votre repo fournit un requirements.txt, lancez:
# !pip install -r requirements.txt

In [ ]:
# Vérifier Python / Torch / GPU
import sys, torch
print('Python', sys.version.split()[0])
print('Torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))

## 5) Configurer chemins & paramètres CLI (modifie si besoin)

In [ ]:
# Modifie ces variables selon ton organisation Drive / repo
DATASET_DIR = '/content/drive/MyDrive/dataset'
SORTIES_DIR = '/content/drive/MyDrive/detection_fissures_sorties'
RESUME_CHECKPOINT = 'sorties/modeles/dernier_modele.pth'
EPOCHS = 100
BATCH_SIZE = 8
LR = 5e-5
DEVICE_ARG = 'cuda'  # ou 'cpu'

# Si le checkpoint existe déjà, on active la reprise automatique
from pathlib import Path
resume_option = f'--resume {RESUME_CHECKPOINT}' if Path(RESUME_CHECKPOINT).exists() else ''

# Exemple d'arguments CLI pour `entrainer.py`
CLI_CMD = (
    f'python entrainer.py --donnees {DATASET_DIR} '
    f'--sorties {SORTIES_DIR} '
    f'--epoques {EPOCHS} '
    f'--lot {BATCH_SIZE} '
    f'--lr {LR} '
    f'--dispositif {DEVICE_ARG} '
    f'{resume_option}'
).strip()
print(CLI_CMD)

## 6) Sélection du dispositif & graine (reproductibilité)

In [ ]:
# Exemple d'utilisation des utilitaires du projet (adapte si fonctions diffèrent)
import sys, os
sys.path.append('/content/detection_fissures')
try:
    from utilitaires import dispositif, graine
    device = dispositif.get_device(prefer='cuda')
    graine.set_seed(42)
    print('Device choisi via utilitaires:', device)
except Exception as e:
    print('Import utilitaires failed — adapte les appels:', e)

## 7) Inspecter le dataset COCO (train/valid/test) — aperçu rapide

In [ ]:
import json, os
from pathlib import Path
def inspect_coco(split_dir):
    ann = Path(split_dir) / '_annotations.coco.json'
    if not ann.exists():
        print(ann, 'n'existe pas')
        return
    d = json.load(open(ann))
    print(split_dir, '— images:', len(d.get('images', [])), 'annotations:', len(d.get('annotations', [])))

for s in ['train', 'valid', 'test']:
    inspect_coco(os.path.join(DATASET_DIR, s))

## 8) Lancer l'entraînement (exécute `entrainer.py`)

In [ ]:
# Lance la commande d'entraînement définie plus haut
print('Lancement:', CLI_CMD)
!$CLI_CMD

## 9) Exemple: reprendre un checkpoint et continuer l'entraînement

In [ ]:
# Exemple shell pour reprendre depuis un checkpoint (adapte selon le code du projet)
# !python entrainer.py --donnees {DATASET_DIR} --resume sorties/modeles/checkpoint_last.pth --epoques 50 --dispositif cuda

## 10) Validation, métriques et visualisations (placeholders)

Les sections suivantes sont des points de départ — adapte-les au code du projet:
- Construction des DataLoader (`donnees/jeu_donnees_coco.py`, `donnees/chargeur.py`).
- Initialisation du modèle (`modeles/masque_rcnn.py`).
- Perte et métriques (`entrainement/pertes.py`, `entrainement/metriques.py`).
- Visualisation des prédictions (matplotlib + masks).
- Exécution de `analyser.py` pour classification des fissures.

## 11) Export & inference single-image (exemple)

In [ ]:
# Exemple: charger un modèle sauvegardé et faire une inférence sur une image
# from modeles.masque_rcnn import build_model
# model = build_model(num_classes=2, pretrained=False)
# checkpoint = torch.load('sorties/modeles/meilleur_modele.pth', map_location='cpu')
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()
# TODO: ajouter préprocessing puis inference et affichage

## 12) Hyperparam sweep / tests / utilitaires
- Utilise des boucles rapides pour tester quelques combos `lr` / `batch_size` / `epochs`.
- Pour tests unitaires: `!pytest -q` si tu as des tests dans le repo.

---
Notes finales: ouvre ce notebook dans Colab (File → Upload notebook) ou depuis GitHub.
Dis‑moi si tu veux que je remplisse `GITHUB_REPO_URL` et `DATASET_DIR` automatiquement et je le fais.